# PRISM steps 11-13: audit, plan, approve, verify

Starts from the schema saved with notebook 05 (and the confirmations saved with notebook 07). See `docs/GUIDE.md` for what every step does.

Nothing changes your data until you approve a plan entry; every approved entry makes a new data state in `work/states/`, and the audit is re-run on it.

In [ ]:
from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q --force-reinstall --no-deps "git+https://{token}@github.com/Isra-inventor/PRISM.git@claude/prism-data-input-recognition-9adu1r"
import prism
print("PRISM", prism.__version__)
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/PRISM_data"
OUT = "/content/drive/MyDrive/PRISM_out"

## 1. The full audit (checkpoint C0)

In [ ]:
s = prism.load(f"{OUT}/soma/schema.json", prism.read(f"{DATA}/SomaExpr_common.csv"))
st = prism.Study(s, overrides=f"{OUT}/soma/overrides.json", folder=f"{OUT}/soma/work")
st.audit()

In [ ]:
st.root_audit          # every finding, with its sentences

## 2. What only you can answer

In [ ]:
st.questions()

In [ ]:
# st.confirm_processing_state("normalized", "yes", assay="SomaScan")
# st.answer_question("floor_censored", "no")

## 3. The intake: what your study compares

The sample variables PRISM knows:

In [ ]:
st.variable_names()

In [ ]:
st.set_intake(contrast_variable="none", question_type="global_untargeted", downstream_family="unsure")   # change to yours

## 4. The plan

Every step with its status (Recommended, Alternative, Do not automate, Insufficient metadata) and its candidates. Read the reasons before approving.

In [ ]:
plan = st.build_plan()
plan

In [ ]:
plan.d["gate"][0]["steps"]       # which steps lack which metadata

## 5. Approve one entry at a time, rebuilding the plan after each

```python
st.approve("A1B1:G3:flagged:Type")   # e.g. remove the flagged controls
plan = st.build_plan(); plan
st.approve("A1B1:G5")                # e.g. the log transform
plan = st.build_plan(); plan
```
To say no: `st.reject("A1B1:S1", reason="...")`.

In [ ]:
# st.approve("...")
st.states()

## 6. Verify (at C2), replay, save

In [ ]:
# st.verify()          # after a batch correction or a filtering step
# st.compare()         # S0 against now, same measures
st.replay()

In [ ]:
st.save()